[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-14-multilingual.ipynb)


# Lecture 14: one model, both directions

Until now each of your models has translated one way: A9 from English into your language and,
if you trained one in A13, a reverse model back. A multilingual model does both, and more, with
one set of weights. The trick is small: a **tag** at the start of each sequence says which
language is which, and the model trains on both directions' pairs mixed together. Meta's
NLLB-200 is that trick at full scale: one model, about 200 languages, steered by language codes.

That is Assignment 14. You **fine-tune** NLLB-200 on your A9 split, both directions in one run,
and ask two questions of the result: did one model translating both ways match your one-way
models, and what did starting from a pretrained model buy over starting from scratch, as A9 did?

By the end you will be able to:

1. tag each pair with the language to produce and intermingle both directions into one training
   set, keeping both directions' test sentences out of it;
2. explain how a tag tells one model which way to translate, and why tagging lets a multilingual
   model translate between languages it never saw paired (zero-shot);
3. compare one model that translates both ways with two one-way models, with decoding held
   fixed and the difference tested;
4. measure what fine-tuning a pretrained model buys over the same model off the shelf, and over
   a model trained from scratch.

These serve Lecture 14's outcome *explain how a tag tells one model which language to write;
explain why a tagged model can translate pairs it never trained on; explain how a
low-resource language borrows from related ones in one model; explain when fine-tuning a
pretrained model beats training from scratch; explain how Assignment 14 compares one two-way and two one-way models*.

You choose your language's code, write the tagging and the intermingling, state a hypothesis
and a prediction, set the training budget, and choose a third language for the zero-shot test;
checks tell you when each is right, and the notebook runs the rest. Tutorial 9, after it, asks
how much fine-tuning is enough and what it costs; this notebook does not need it.

**What you need.** Your A9 files in `MyDrive/CS479/assignment8`, as Lecture 9's Part B left
them: the split, `spm.model`, `a9-settings.json` and `best-a9/`. For the comparison into
English, your A13 reverse model, `best-a13-reverse/`, if you trained one. About **8 GB free on
your Drive** for the training's checkpoints.

**This is long.** Use an A100, L4 or G4, not a T4. The training prints how long your budget
will take after its first 20 steps; if that is more than you have, stop it, lower `STEPS` and
run it again, since it resumes. Every step saves to Drive and can be picked up in a new
session: run the setup cells, then the step you were on. **When you finish, disconnect the
GPU**: Runtime → Disconnect and delete runtime. Not while the training or a decoding cell is
running; the last cell says why.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — homework for Lecture 14 (*Multilingual NMT and "Zero-shot" NMT*). A head start on **A14**.

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo


In [ ]:
# Setup: the GPU, Drive, your A9 files, and the helpers the steps below share. Run it first in
# every session.
import json
import textwrap

import pandas as pd
import torch
import torchlingo
from torchlingo.checkpoint import load_model
from torchlingo.colab import Notes, course_dir, save_rerun, setup
from torchlingo.config import Config
from torchlingo.data_processing.vocab import SentencePieceVocab
from torchlingo.evaluation import compute_chrf, format_scores, paired_bootstrap
from torchlingo.inference import translate_lines_to_file, translate_to_file
from torchlingo.training_checkpoint import TrainingCheckpointer

env = setup(gpu=True, drive=True)       # stops with instructions if there is no GPU
device = env.device
# Stops, naming each missing file, if A9 has not left them in MyDrive/CS479/assignment8.
A8_DIR = course_dir(
    env.drive_dir, "assignment8",
    needs=["train.tsv", "val.tsv", "test.tsv", "spm.model", "a9-settings.json",
           "best-a9/model_best.pt"],
    hint="Lecture 9's Part B writes them; finish A9 first.",
)

settings = json.loads((A8_DIR / "a9-settings.json").read_text(encoding="utf-8"))
NO_SPACES = settings["no_spaces"]
TOKENIZE = "flores200" if NO_SPACES else "13a"     # the BLEU A9 was scored with
SEED = 479
ENG = "eng_Latn"                                   # NLLB's code for English
CHECKPOINTS = A8_DIR / "a14-checkpoints"
# What each step saves for the write-up, in a14-notes.json on Drive, so a dropped session loses
# none of it: save_note(name=value) adds to it, load_notes() reads it all back.
NOTES = Notes(A8_DIR / "a14-notes.json")
load_notes, save_note = NOTES.load, NOTES.save


def read_tsv(name):
    return pd.read_csv(A8_DIR / name, sep="\t", dtype=str, keep_default_na=False)


# Your own one-way models, trained from scratch: A9 into your language and, if A13 trained one,
# the reverse model into English. They decode as A9 and A13 decoded them.
vocab = SentencePieceVocab(str(A8_DIR / "spm.model"))
config = Config(d_model=512, n_heads=8, num_encoder_layers=6, num_decoder_layers=6, d_ff=2048,
                batch_size=64, num_workers=0, max_decode_length=settings["max_decode_length"])
INTO_ENGLISH = "A13 reverse" if (A8_DIR / "best-a13-reverse" / "model_best.pt").exists() else None


def scratch_translate(best_dir, sentences, path, decoder):
    """Translate with one of your TorchLingo models, resuming a decode a dropped session cut."""
    # config builds the model only if its file is too old to record how it was built.
    model = load_model(A8_DIR / best_dir / "model_best.pt", device, config=config,
                       src_vocab=vocab).model
    return translate_to_file(model, sentences, vocab, vocab, path, decode_strategy=decoder,
                             beam_size=5, device=device, config=config, verbose=False)


train_df, val_df, test_df = read_tsv("train.tsv"), read_tsv("val.tsv"), read_tsv("test.tsv")
print(f"Your split: {len(train_df):,} train, {len(val_df):,} validation, {len(test_df):,} test pairs.")
print(f"Into English, your one-way model: {INTO_ENGLISH or 'none (no A13 reverse model)'}")


In [ ]:
# Setup: run this cell, no need to read it. It loads NLLB-200 (about 2.4 GB, a minute or two
# the first time) and the helpers that decode with it.
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

from torchlingo_harness.nllb import NLLB_MODEL, NLLBTranslator

tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL, dtype=torch.float32).to(device)
NLLB_CODES = {t for t in tokenizer.additional_special_tokens if len(t) == 8 and t[3] == "_"}
FINE_TUNED = False   # becomes True once the model in memory holds the fine-tuned weights


def nllb_translate(sentences, src_code, tgt_code, path, decoder):
    """Translate with NLLB as it is in memory, from src_code into tgt_code, to a file on Drive."""
    tokenizer.src_lang = src_code
    nllb = NLLBTranslator(model.eval(), tokenizer, tgt_code, device, NLLB_MODEL)
    options = {"decode_strategy": decoder, "beam_size": 5, "batch_size": 16}
    return translate_lines_to_file(lambda chunk, first: nllb(chunk, **options), sentences,
                                   path, nllb.settings(**options), verbose=False)


def use_fine_tuned():
    """Put the fine-tuned weights in memory, from Drive, if this session has not got them."""
    global FINE_TUNED
    if not FINE_TUNED:
        run = TrainingCheckpointer("assignment-14", checkpoint_dir=CHECKPOINTS, verbose=False)
        assert run.has_checkpoint("best"), "Train the model in Step 4 first."
        run.load(model, which="best", map_location=device, restore_rng=False)
        FINE_TUNED = True


print(f"{NLLB_MODEL}: {sum(p.numel() for p in model.parameters()):,} parameters, "
      f"{len(NLLB_CODES)} language codes")


## Step 1: your language's code

NLLB's tag is a **language code**: three letters for the language, four for its script, as in
`eng_Latn`, `deu_Latn` or `zho_Hant`. Every source sentence starts with its own language's
code, and every target with the code of the language to produce, so a pair is
`eng_Latn Good morning </s>` → `deu_Latn Guten Morgen </s>`. When the model translates, its
decoder is handed the target code as its first word and continues from there: that one code is
all that says which way to go. A14 trains both directions, so each pair is used twice, once
with each language's code in front.

**If NLLB covers your language,** its code is yours, and the next cell says so.

**If it does not** (of the A8 languages, Hmong), the model still needs a code to be told what
to write, so you **borrow** a covered language's, and fine-tuning teaches the model that the
code now means yours. A related language's code starts the model nearer your language's
script, spelling and word order. Reason it out first: language family, script, shared
vocabulary or loanwords, word order, and the languages of your language's region and contact.
Then list two or three candidates in `CANDIDATES` (codes from the
[FLORES-200 list](https://github.com/facebookresearch/flores/blob/main/flores200/README.md)):
the cell translates a dozen of your test sentences into each, off the shelf, and prints chrF
against your references. Choose in the cell after, and say why in `REASON`.

In [ ]:
LANGUAGE = ""     # <- your A8 language, as Learning Suite's data folder names it, e.g. "German"
CANDIDATES = []   # <- only if NLLB does not cover it: two or three codes, e.g. ["abc_Latn", ...]

# The A8 languages and their NLLB codes; None where NLLB does not cover the language.
COURSE_CODES = {
    "Chinese T": "zho_Hant", "French": "fra_Latn", "Georgian": "kat_Geor", "German": "deu_Latn",
    "Greek": "ell_Grek", "Hmong": None, "Japanese": "jpn_Jpan", "Khmer": "khm_Khmr",
    "Korean": "kor_Hang", "Lao": "lao_Laoo", "Malagasy": "plt_Latn", "Malay": "zsm_Latn",
    "Norwegian": "nob_Latn", "Portuguese": "por_Latn", "Romanian": "ron_Latn",
    "Russian": "rus_Cyrl", "Spanish": "spa_Latn", "Tagalog": "tgl_Latn", "Thai": "tha_Thai",
}
assert LANGUAGE in COURSE_CODES, f"Set LANGUAGE to one of {sorted(COURSE_CODES)}."
COVERED = COURSE_CODES[LANGUAGE]

if COVERED:
    print(f"NLLB covers {LANGUAGE}: its code is {COVERED}. Nothing to borrow.")
else:
    assert CANDIDATES, (f"NLLB does not cover {LANGUAGE}. List two or three candidate codes in "
                        "CANDIDATES, as above, then run this cell again.")
    unknown = [c for c in CANDIDATES if c not in NLLB_CODES]
    assert not unknown, f"not NLLB codes: {unknown}; codes look like 'tgl_Latn'"
    dozen = test_df.sample(12, random_state=SEED)
    print("chrF of each candidate, off the shelf, on 12 of your test sentences:\n")
    for code in CANDIDATES:
        tokenizer.src_lang = ENG
        hyps = NLLBTranslator(model.eval(), tokenizer, code, device, NLLB_MODEL)(list(dozen.src))
        print(f"  {code}   chrF {compute_chrf(hyps, list(dozen.tgt)).score:5.1f}   e.g. {hyps[0][:60]!r}")
    print(f"\n  reference:          {dozen.tgt.iloc[0][:60]!r}")


In [ ]:
BORROW = ""   # <- only if NLLB does not cover your language: the code you borrow
REASON = ""   # <- why: family, script, vocabulary, word order, region; and what chrF said

if COVERED:
    X_CODE, REASON = COVERED, f"{LANGUAGE} is one of NLLB's languages"
else:
    assert BORROW in CANDIDATES, "Set BORROW to one of your CANDIDATES, then run this cell again."
    assert REASON.strip(), "Say in REASON why you chose it, then run this cell again."
    X_CODE = BORROW
save_note(language=LANGUAGE, x_code=X_CODE, reason=REASON.strip())
print(f"Your language's code: {X_CODE}  ({REASON.strip()})")


## Step 2: tag, intermingle, and keep the test sentences out

**Yours to write: `tag` and `both_directions`.** Each training pair carries its two codes in
columns of its own, `src_code` and `tgt_code`; the training puts them in front of the sentences.

- `tag(frame, src_code, tgt_code)` returns the same pairs with those two columns added, in the
  order `src`, `tgt`, `src_code`, `tgt_code`, leaving the table it was given unchanged.
- `both_directions(frame)` returns every pair twice: English to your language, tagged
  `ENG` → `X_CODE`, and the same pair turned around, your sentence to its English, tagged
  `X_CODE` → `ENG`. Then it **intermingles** the two halves by shuffling them with `SEED`.
  Trained on one direction and then the other, a model drifts toward whichever came last;
  mixed, every batch holds both.

The cell checks both on a known case, then builds `train-a14.tsv` and `val-a14.tsv` from A9's
training and validation sets, and one test set per direction from A9's test set,
`test-a14-to-x.tsv` and `test-a14-to-en.tsv`. Every test sentence is now a source in one
direction and a reference in the other, so the contamination check runs on both languages: no
test sentence, English or yours, may be anywhere in training.

In [ ]:
from torchlingo.diagnostics import check_contamination


def tag(frame, src_code, tgt_code):
    """Return the same pairs with columns src_code and tgt_code added.

    TODO: write it, in a line or two.
    """
    raise NotImplementedError("write tag, then run this cell again")


def both_directions(frame):
    """Return every pair once each way, tagged with its codes, intermingled.

    TODO: write it, as the markdown above describes, using tag.
    """
    raise NotImplementedError("write both_directions, then run this cell again")


# A case whose answer you know: two pairs, with Spanish's code standing in for yours.
COLUMNS = ["src", "tgt", "src_code", "tgt_code"]
toy = pd.DataFrame({"src": ["good morning", "thank you"], "tgt": ["buenos días", "gracias"]})
tagged = tag(toy, ENG, "spa_Latn")
expected = [["good morning", "buenos días", ENG, "spa_Latn"], ["thank you", "gracias", ENG, "spa_Latn"]]
assert list(tagged.columns) == COLUMNS, f"the columns must be {COLUMNS}, in that order; yours are {list(tagged.columns)}"
assert tagged.values.tolist() == expected, (
    f"tag is not right yet: for this case it should return {expected}, and yours returned "
    f"{tagged.values.tolist()}")
assert list(toy.columns) == ["src", "tgt"], "tag changed the table it was given; work on a copy"

both = both_directions(toy)
expected = sorted([("good morning", "buenos días", ENG, X_CODE), ("thank you", "gracias", ENG, X_CODE),
                   ("buenos días", "good morning", X_CODE, ENG), ("gracias", "thank you", X_CODE, ENG)])
assert list(both.columns) == COLUMNS, f"the columns must be {COLUMNS}, in that order"
assert sorted(map(tuple, both.values.tolist())) == expected, (
    f"both_directions is not right yet: for this case it should return these four rows, in "
    f"any order, {expected}, and yours returned {both.values.tolist()}")
assert both.equals(both_directions(toy)), "the same SEED must give the same order every time"

# Your split, both ways. The first half must hold both directions, or they are not intermingled.
combined = both_directions(train_df[["src", "tgt"]])
share = (combined.tgt_code.iloc[:len(combined) // 2] == X_CODE).mean()
assert 0.2 < share < 0.8, (
    f"{share:.0%} of the first half of the training set goes into your language: the two "
    "directions are not intermingled. Shuffle the pairs after putting them together.")
tests = {
    "to-x": tag(test_df[["src", "tgt"]], ENG, X_CODE),
    "to-en": tag(test_df.rename(columns={"src": "tgt", "tgt": "src"})[["src", "tgt"]], X_CODE, ENG),
}

# No test sentence, in either language, may be anywhere in training, source or target.
in_training = list(combined.src) + list(combined.tgt)
for language, sentences in (("English", test_df.src), ("your language", test_df.tgt)):
    result = check_contamination(sentences, in_training, name=f"test, {language}")
    print(result)
    assert result, "a test sentence is in the training set; it would put the answer into training"

combined.to_csv(A8_DIR / "train-a14.tsv", sep="\t", index=False)
both_directions(val_df[["src", "tgt"]]).to_csv(A8_DIR / "val-a14.tsv", sep="\t", index=False)
for name, frame in tests.items():
    frame.to_csv(A8_DIR / f"test-a14-{name}.tsv", sep="\t", index=False)
save_note(training_pairs=len(combined))
print(f"Written: train-a14.tsv, {len(combined):,} pairs from A9's {len(train_df):,}; val-a14.tsv; "
      "test-a14-to-x.tsv and test-a14-to-en.tsv.")
print(f"For example: {combined.src_code.iloc[0]} {combined.src.iloc[0]}  ->  "
      f"{combined.tgt_code.iloc[0]} {combined.tgt.iloc[0]}")


## Step 3: the numbers to beat

Before training, NLLB **off the shelf** translates both test sets, greedily and with a beam of
5: what the pretrained model already knows, and so the baseline for what fine-tuning buys. For
a borrowed code, "into your language" off the shelf is really into the borrowed one, and
should score low. Everything is saved to Drive, and a decode cut off by a dropped session
resumes when you run the cell again. Run this before Step 4: afterwards the model in memory is
the fine-tuned one (in a new session, the setup cells load the original again).

In [ ]:
DIRECTIONS = {   # each direction: its test set, its codes, its references and its BLEU
    "to-x": {"name": "into your language", "codes": (ENG, X_CODE), "sources": list(test_df.src),
             "refs": list(test_df.tgt), "tokenize": TOKENIZE, "one_way": "A9"},
    "to-en": {"name": "into English", "codes": (X_CODE, ENG), "sources": list(test_df.tgt),
              "refs": list(test_df.src), "tokenize": "13a", "one_way": INTO_ENGLISH},
}
DECODERS = ("greedy", "beam")
hyps, hyp_files = {}, {}


def keep(system, direction, decoder, path, lines):
    hyps[system, direction, decoder] = lines
    hyp_files[f"{system}|{direction}|{decoder}"] = path.name


for direction, d in DIRECTIONS.items():
    for decoder in DECODERS:
        path = A8_DIR / f"test-nllb-{direction}-{decoder}.hyp"
        assert not FINE_TUNED or path.exists(), (
            "The model in memory is fine-tuned now. To decode NLLB off the shelf, run the "
            "setup cells in a new session, then this cell.")
        print(f"NLLB off the shelf, {d['name']}, {decoder}: {path.name}")
        keep("NLLB off the shelf", direction, decoder, path,
             nllb_translate(d["sources"], *d["codes"], path, decoder))
        print(f"   chrF {compute_chrf(hyps['NLLB off the shelf', direction, decoder], d['refs']).score:.2f}")
save_note(hyp_files={**load_notes().get("hyp_files", {}), **hyp_files})


## Step 4: a hypothesis, a prediction, a budget; then fine-tune

Before the model trains, **state your hypothesis**, the Final Project's Path A in small:
*"If I fine-tune NLLB on my pairs in both directions, then its BLEU into my language will ..."*,
finished with what you expect against A9 and why. Then **a prediction**: roughly where each
direction will land against NLLB off the shelf and against your one-way model. The notebook
stops until you write both, and saves them to Drive for the write-up.

**The budget is yours too.** Fine-tuning is measured in **steps**, each one update of the
weights from 32 pairs. The default, 3,000 steps, is 96,000 pairs, about half a pass over
A14's training set, which holds each A9 pair twice. More steps cost time; past some point they
buy nothing, and the validation loss, checked every 500 steps, says when.

The training is the deep-learning part, and the library runs it. What it does, so you can
read what it prints: it trains **every weight** of the pretrained model a little, at a learning
rate far smaller than A8's, so as to nudge what 200 languages taught it rather than overwrite
it, and raises that rate from zero over the first 100 steps (**warmup**), so the first updates,
made before the model has adapted to your pairs, cannot knock it far; it adds up the gradients
of two batches of 16 before each update (**accumulation**, the update of a batch of 32 in the
memory of 16); it does the arithmetic in **bfloat16** where the GPU has it, about twice as fast;
and it uses **Adafactor**, an optimizer whose saved state is small, so each checkpoint on your
Drive is about the size of the weights, 2.5 GB. It keeps the checkpoint with the best
validation loss, as A8 and A9 did.

In [ ]:
HYPOTHESIS = ""   # <- "If I fine-tune NLLB on my pairs in both directions, then ... will ..."
PREDICTION = ""   # <- where each direction will land against NLLB off the shelf and your one-way model
STEPS = 3000      # <- your budget: about half a pass over A14's training set

assert HYPOTHESIS.strip(), "Write your hypothesis in HYPOTHESIS, then run this cell again."
assert HYPOTHESIS.strip().lower().startswith("if") and " then " in HYPOTHESIS.lower(), (
    'Write the hypothesis as "If ..., then ...": what you change, and what you expect it to do.')
assert PREDICTION.strip(), "Write your prediction in PREDICTION, then run this cell again."
save_note(hypothesis=HYPOTHESIS.strip(), prediction=PREDICTION.strip(), steps=STEPS)
print(f"Saved to a14-notes.json. {STEPS:,} steps of 32 pairs: "
      f"{STEPS * 32 / (2 * len(train_df)):.2f} passes over A14's training set.")


In [ ]:
# The long training: NLLB, both directions at once. If the session drops, run the setup cells
# and this one again: an unfinished training resumes, a finished one is skipped.
from torchlingo_harness.nllb import fine_tune

state = fine_tune(model, tokenizer, read_tsv("train-a14.tsv"), read_tsv("val-a14.tsv"),
                  TrainingCheckpointer("assignment-14", checkpoint_dir=CHECKPOINTS),
                  steps=STEPS, device=device, eval_every=500, seed=SEED)
FINE_TUNED = True
save_note(steps_taken=state.global_step, val_losses=[round(v, 4) for v in state.val_losses])
print(f"Fine-tuned for {state.global_step:,} steps; best validation loss {state.best_val_loss:.3f}.")


## Step 5: score each direction

The fine-tuned model translates both test sets, and every system decodes **greedily and with
beam search (k = 5)**; a comparison is only ever between two systems decoded the same way.
Greedy takes the likeliest next piece at each step; beam search keeps the five likeliest
partial translations at each step and returns the best finished one. Each direction has three
systems: **A14**, NLLB fine-tuned on your pairs; **NLLB off the shelf**, from Step 3; and **your
one-way model**, trained from scratch: A9 into your language, and your A13 reverse model into
English, if you have one. Into your language the BLEU is A9's; into English it is SacreBLEU's
default; and chrF for both, which counts matching characters rather than whole words.

**sacreBLEU's paired bootstrap** resamples your test set a thousand times and scores every
system on each resample. It gives each score a 95% interval and, for each other system, a *p*
against A14: roughly, how often a difference this large would appear by the luck of which
sentences landed in the test set. Below 0.05 is the usual line.

**Then say what the table says.** In the cell after it, set `REAL` to whether each greedy
difference from A14 is real; the cell prints the comparisons to answer, and checks you against
the *p*.

A9's translations from A13 are reused if they are on Drive. Everything is saved to Drive as it
goes, and a decode a dropped session cut off resumes.

In [ ]:
for key, name in load_notes().get("hyp_files", {}).items():   # Step 3's, from Drive
    system, direction, decoder = key.split("|")
    keep(system, direction, decoder, A8_DIR / name,
         (A8_DIR / name).read_text(encoding="utf-8").split("\n")[:len(test_df)])
use_fine_tuned()
for direction, d in DIRECTIONS.items():
    for decoder in DECODERS:
        path = A8_DIR / f"test-a14-{direction}-{decoder}.hyp"
        print(f"A14, {d['name']}, {decoder}: {path.name}")
        keep("A14", direction, decoder, path, nllb_translate(d["sources"], *d["codes"], path, decoder))
        if d["one_way"]:
            name = "a9" if d["one_way"] == "A9" else "a13-reverse"
            path = A8_DIR / f"test-{name}-{decoder}.hyp"
            print(f"{d['one_way']}, {d['name']}, {decoder}: {path.name}")
            keep(d["one_way"], direction, decoder, path,
                 scratch_translate(f"best-{name}", d["sources"], path, decoder))


def bootstrap(direction, decoder):
    """BLEU and chrF for each system with a 95% interval, and each other system's p against A14,
    from sacreBLEU's paired bootstrap over 1,000 resamples of the test set."""
    d = DIRECTIONS[direction]
    names = [n for n in ("A14", "NLLB off the shelf", d["one_way"]) if n]
    return paired_bootstrap({n: hyps[n, direction, decoder] for n in names}, d["refs"],
                            tokenize=d["tokenize"])


scores = {direction: {decoder: bootstrap(direction, decoder) for decoder in DECODERS}
          for direction in DIRECTIONS}
save_note(scores=scores, hyp_files=hyp_files, into_english=INTO_ENGLISH)
for direction, d in DIRECTIONS.items():
    for decoder in DECODERS:
        print(f"\n{d['name']}, {decoder}:")
        print(textwrap.indent(format_scores(scores[direction][decoder]), "    "))


In [ ]:
REAL = {}   # <- each comparison this cell prints, decoded greedily: True if the difference is real, else False

names = {"to-x": "into your language", "to-en": "into English"}
p_values = {f"{names[direction]}: A14 against {system}": s["bleu_p"]
            for direction, rows in load_notes()["scores"].items()
            for system, s in rows["greedy"].items() if system != "A14"}
if set(REAL) != set(p_values):
    print("Copy these into REAL, each with True or False, read off the greedy rows above:")
    for comparison in p_values:
        print(f'    "{comparison}": None,')
assert set(REAL) == set(p_values) and None not in REAL.values(), (
    "Set REAL to the comparisons printed above, each True or False, then run this cell again.")
for comparison, said in REAL.items():
    p = p_values[comparison]
    assert said == (p < 0.05), (
        f"For {comparison} you said {'real' if said else 'not real'}, but its p is {p:.3f}, "
        f"which is {'below' if p < 0.05 else 'not below'} 0.05. Look at that row again.")
save_note(real=REAL)
for comparison, said in REAL.items():
    print(f"{comparison}: {'real' if said else 'not real'} (p = {p_values[comparison]:.3f})")

## Step 6: what the code does, and a pair nobody trained

The target code is the only thing that says which way to go, so give the model the wrong one:
five English sentences asked for English, and the same five in your language asked for your
language. Does it copy, paraphrase, translate anyway, or produce something else?

Then **zero-shot**: the same five sentences in your language, asked for a **third language**
you can read. Your pairs never held that third language, and NLLB may never have seen your
language paired with it, or, for a borrowed code, seen your language at all. Choose one NLLB
covers, in `THIRD`.

**Predict first**, in `EXPECT`: for each of the three, what will come out? Copy, paraphrase,
the other language anyway, or something else, and for the third language, whether the output
will be in it at all and keep the meaning. The cell stops until you write it, and the write-up
asks you to set it beside what came out.

This is also where A14's stretch task starts: give the opposite code on twenty test sentences
and describe what comes out.

In [ ]:
THIRD = ""    # <- a third language you can read, as an NLLB code, e.g. "spa_Latn" or "fra_Latn"
EXPECT = ""   # <- what you expect from each of the three, before you see it

assert THIRD in NLLB_CODES and THIRD not in (ENG, X_CODE), (
    "Set THIRD to the NLLB code of a third language you can read, not English or yours.")
assert EXPECT.strip(), "Write what you expect in EXPECT, then run this cell again."
use_fine_tuned()
picked = test_df.head(5)
cases = {
    "English, asked for English": (list(picked.src), ENG, ENG),
    "your language, asked for your language": (list(picked.tgt), X_CODE, X_CODE),
    f"your language, asked for {THIRD}": (list(picked.tgt), X_CODE, THIRD),
}
shown = {}
for title, (sentences, src_code, tgt_code) in cases.items():
    tokenizer.src_lang = src_code
    out = NLLBTranslator(model.eval(), tokenizer, tgt_code, device, NLLB_MODEL)(sentences)
    shown[title] = [[s, o] for s, o in zip(sentences, out)]
    print(f"=== {title} ===")
    for (source, output), english in zip(shown[title], picked.src):
        print(f"  in:  {source}\n  out: {output}" + (f"\n  (English: {english})" if src_code != ENG else ""))
print(f"\nYou expected: {EXPECT.strip()}")
save_note(wrong_code=shown, third=THIRD, expected=EXPECT.strip())


## Step 7: COMET

The reference-based COMET model you installed for A10, `Unbabel/wmt22-comet-da`, scores every
translation from Steps 3 and 5: each system, each direction, greedy and beam. It needs no
Hugging Face token. The first cell installs it, the second scores and saves the system scores
for the write-up. They are the last to run before the write-up, because installing COMET can
change packages the cells above rely on; if they give trouble, run them in a fresh session on
their own, after the setup cells.

**Read COMET into your language with care.** COMET reads your language through its own
multilingual encoder, so its scores into your language mean less if that encoder saw little of
it. Into English, COMET is on its firmest ground.

In [ ]:
# COMET asks for numpy older than 2, which would break packages Colab ships built for numpy 2;
# COMET itself runs on numpy 2. So its other requirements go in first, then COMET without them.
# pip may then print a red list of conflicts with other packages Colab ships; this uses none.
%pip install --quiet "entmax>=1.1,<2" "jsonargparse==3.13.1" "pytorch-lightning>=2,<3" "sacrebleu>=2,<3" "setuptools<81" "torchmetrics>=0.10.2,<0.11" "transformers>=4.17,<5"
%pip install --quiet --no-deps unbabel-comet
from importlib.metadata import version
if int(version("numpy").split(".")[0]) < 2:
    raise RuntimeError("An earlier COMET install replaced numpy in this session, which breaks COMET. "
                       "Runtime -> Disconnect and delete runtime, then run the setup cells and this one.")


In [ ]:
# Every translation from Steps 3 and 5, read from Drive, scored against its references.
from torchlingo.evaluation import comet_score   # loads Unbabel/wmt22-comet-da the first time

comet_scores = {}
for key, name in load_notes()["hyp_files"].items():
    system, direction, decoder = key.split("|")
    d = DIRECTIONS[direction]
    mt = (A8_DIR / name).read_text(encoding="utf-8").split("\n")[:len(d["refs"])]
    score = comet_score(d["sources"], mt, d["refs"]).system
    comet_scores.setdefault(direction, {}).setdefault(decoder, {})[system] = round(score, 4)
    print(f"{d['name']:<19} {decoder:<6} {system:<19} COMET {score:.4f}")
save_note(comet=comet_scores)


## Step 8: the write-up

The cell below prints the numbers in the order the write-up asks for and saves `a14-rerun.json`.
It reads everything from Drive, so in a new session it needs only the setup cell. Answer these in
your write-up:

1. **Your hypothesis against the result.** Quote your "If ..., then ..." and your prediction,
   then give BLEU and chrF, with their intervals, for each direction and each system, greedy
   and beam, and the COMET system scores. Did the paired bootstrap find each difference from
   A14 real (p below 0.05)? Does COMET agree with BLEU about each difference?
2. **What did fine-tuning buy over NLLB off the shelf, in each direction?** If you borrowed a
   code, say how much of the off-the-shelf score into your language was the borrowed language,
   and how far fine-tuning moved the model from it to yours.
3. **Into your language: fine-tuned NLLB against A9.** Both learned from your pairs; one started
   from 200 languages and 615 million weights, the other from nothing and about 56 million.
   Which won, by how much, and which of those differences do you think the result reflects?
4. **Into English: A14 against your A13 reverse model.** If you had none, say what you would
   expect and why, from the other direction's result.
5. **How does one model know which way to translate?** Use Step 1's codes and Step 6's
   wrong-code outputs, set beside what you expected: what did your model learn the code means,
   and what did it do when the code asked for the language it was given?
6. **Why does zero-shot work at all?** Use Step 6's third-language outputs: did the model write
   the third language, and did it keep the meaning? Say what the model must share across
   languages for that to work, and why it may still answer in the wrong language.
7. **How would someone rerun this?** Paste the contents of `a14-rerun.json`.

**Upload** to your folder in the Assignment shared folder: `train-a14.tsv`, the two test sets
`test-a14-to-x.tsv` and `test-a14-to-en.tsv`, every `.hyp` file Steps 3 and 5 wrote,
`a14-rerun.json`, the write-up, and **this notebook**: File → Download → Download .ipynb, after
it has run, so every cell's output is in it.

In [ ]:
# Reads everything from Drive, so after a dropped session it needs only the setup cell above.
notes = load_notes()
names = {"to-x": "into your language", "to-en": "into English"}
which = "spBLEU (flores200)" if NO_SPACES else "SacreBLEU, default tokenizer"

print("=== For the A14 write-up ===")
print(f"Hypothesis, written before training: {notes['hypothesis']}")
print(f"Prediction, written before training: {notes['prediction']}\n")
print(f"Codes: {ENG} for English, {notes['x_code']} for {notes['language']} ({notes['reason']})")
print(f"Training pairs: {notes['training_pairs']:,}, A9's split once each way; "
      f"{notes.get('steps_taken', 0):,} steps; validation losses {notes.get('val_losses')}")
print(f"Into English, your one-way model: {notes['into_english'] or 'none (no A13 reverse model)'}\n")

print(f"BLEU into your language is {which}; into English, SacreBLEU's default. Each score is "
      "followed by its 95% interval; then COMET; then each other system's p against A14, "
      "decoded the same way:")
for direction, by_decoder in notes["scores"].items():
    for decoder, rows in by_decoder.items():
        print(f"\n  {names[direction]}, {decoder}:")
        comet = notes.get("comet", {}).get(direction, {}).get(decoder, {})
        print(textwrap.indent(format_scores(rows, comet=comet), "    "))

for title, rows in notes.get("wrong_code", {}).items():
    print(f"\n=== {title} ===")
    for source, output in rows:
        print(f"  in:  {source}\n  out: {output}")

# Everything someone needs to rerun A14 and get the same numbers; save_rerun adds the versions
# of TorchLingo and the libraries that produced them.
save_rerun(
    A8_DIR / "a14-rerun.json",
    model=NLLB_MODEL, codes={"English": ENG, notes["language"]: notes["x_code"]},
    seed=SEED, steps=notes.get("steps_taken"),
    training="A9's train.tsv once each way with its codes, shuffled with the seed; "
             "torchlingo_harness.nllb.fine_tune defaults: Adafactor lr 1e-4, warmup 100, "
             "batch 16 x 2 accumulated, bfloat16 where available, best by validation loss",
    scoring=f"greedy, and beam 5; into your language {which}, into English SacreBLEU "
            "default; chrF; paired bootstrap, 1,000 samples; COMET Unbabel/wmt22-comet-da",
    third_language=notes.get("third"), into_english_one_way=notes["into_english"],
)
print(f"\nSaved {A8_DIR / 'a14-rerun.json'}: paste it into the write-up (question 7).")


---
## When you finish: disconnect the GPU

**Runtime → Disconnect and delete runtime.** Colab charges compute units for every minute a GPU runtime stays connected, even when nothing is running, so a runtime left open after the work is done uses up your monthly units. Closing the browser tab does not end it right away. **Not while the training or a decoding cell is running**: ending the session stops it (it resumes in a new session, but the time in between is lost). Everything A14 makes is on Drive, so disconnecting at the end loses nothing.